# FullStack IA · Fondation pour l'apprentissage profond

**Objectif :** Descente de gradient, perceptron, PyTorch, convolutions et embeddings.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Vecteurs et matrices

En ML, tout est vecteur ou matrice. Un exemple (un manchot, un passager) est un **vecteur** de caractéristiques. Un jeu de données est une **matrice** X : une ligne par exemple, une colonne par caractéristique.

## Vecteurs

In [ ]:
import numpy as np
u = np.array([3.0, 4.0])
v = np.array([1.0, 2.0])
print(u + v, 2 * u)
print("norme de u :", np.linalg.norm(u))     # longueur : racine(3² + 4²) = 5

## Le produit scalaire

Le produit scalaire multiplie les composantes deux à deux puis les additionne. C'est l'opération la plus fréquente en ML : une prédiction linéaire est un produit scalaire entre les caractéristiques et les poids.

$$
u \cdot v = \sum_i u_i v_i
$$

In [ ]:
print(u @ v, np.dot(u, v), (u * v).sum())   # trois écritures équivalentes : 3×1 + 4×2 = 11

La **similarité cosinus** compare la direction de deux vecteurs, entre −1 et 1. C'est elle qui mesure la proximité entre deux textes transformés en vecteurs (embeddings) dans un système RAG.

$$
\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}
$$

## Matrices et produit matriciel

Le produit `A @ B` est possible si le nombre de colonnes de A égale le nombre de lignes de B. Une matrice (n, k) multipliée par une (k, m) donne une (n, m).

*Animation interactive « matmul » : à voir sur la plateforme web.*

In [ ]:
X = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])        # 3 exemples, 2 caractéristiques
w = np.array([0.5, -1.0])         # un poids par caractéristique
b = 2.0
predictions = X @ w + b           # (3, 2) @ (2,) → (3,)
print(predictions)

Une seule ligne calcule les prédictions des 3 exemples à la fois. C'est le cœur d'un modèle linéaire et d'une couche de réseau de neurones.

In [ ]:
print(X.T)                 # transposée : lignes et colonnes échangées
print(X.T @ X)             # (2, 3) @ (3, 2) → (2, 2)
print(np.eye(2))           # matrice identité

> **À retenir** : avant chaque produit matriciel, vérifie les formes avec `.shape`. La majorité des bugs en deep learning sont des erreurs de dimensions.

### Exercice 1 · Prédire avec des poids

Un modèle prédit le prix d'un logement : `prix = 3000 × surface + 15000 × pieces + 20000`. Écris les poids dans un vecteur `w`, puis calcule en **une opération matricielle** le vecteur `prix` pour les logements de `X` (colonnes : surface, pièces).

<details><summary>Indice</summary>

w doit avoir autant de valeurs que X a de colonnes. Puis X @ w + b.
</details>

In [ ]:
import numpy as np
X = np.array([[30, 1], [55, 3], [80, 4]], dtype=float)
b = 20000

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert np.allclose(prix, [125000, 230000, 320000]), f"prix vaut {prix}."
print("Bravo, tout est correct.")

### Exercice 2 · Similarité cosinus

Écris `cosinus(u, v)` qui renvoie la similarité cosinus de deux vecteurs NumPy. Puis trouve `plus_proche` : le nom du document dont le vecteur est le plus similaire à `requete`.

<details><summary>Indice</summary>

u @ v divisé par le produit des normes. max(dico, key=dico.get) renvoie la clé de plus grande valeur.
</details>

In [ ]:
import numpy as np
docs = {
    "cours_python": np.array([0.9, 0.1, 0.0]),
    "recette_crepes": np.array([0.0, 0.2, 0.9]),
    "tuto_pandas": np.array([0.7, 0.6, 0.1]),
}
requete = np.array([0.8, 0.5, 0.0])

def cosinus(u, v):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert abs(cosinus(np.array([1.0, 0.0]), np.array([0.0, 1.0]))) < 1e-9, "Deux vecteurs perpendiculaires ont un cosinus de 0."
assert abs(cosinus(np.array([2.0, 2.0]), np.array([1.0, 1.0])) - 1) < 1e-9, "Deux vecteurs de même direction ont un cosinus de 1."
assert plus_proche == "tuto_pandas", f"plus_proche vaut {plus_proche!r}."
print("Bravo, tout est correct.")

---
## Dérivées et gradient

Entraîner un modèle, c'est **minimiser une erreur**. Pour savoir dans quelle direction modifier les paramètres, on utilise la **dérivée**.

## La dérivée : une pente

La dérivée de f en x mesure la pente de la courbe en ce point : de combien f varie quand x augmente un tout petit peu.

$$
f'(x) \approx \frac{f(x + h) - f(x)}{h} \quad \text{pour } h \text{ très petit}
$$

- Si f'(x) > 0, la fonction monte : pour descendre, il faut diminuer x.
- Si f'(x) < 0, elle descend : il faut augmenter x.
- Si f'(x) = 0, on est sur un plat, souvent un minimum ou un maximum.

In [ ]:
def f(x):
    return (x - 3) ** 2

def derivee(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

for x in [0, 2, 3, 5]:
    print(f"x = {x}  pente ≈ {derivee(f, x):.3f}")

La dérivée exacte de (x − 3)² est 2(x − 3) : elle vaut −6 en 0, 0 en 3, et 4 en 5. La pente nulle en x = 3 indique le minimum.

## Les règles à connaître

| Fonction | Dérivée |
| --- | --- |
| c (constante) | 0 |
| x^n | n·x^(n−1) |
| a·x + b | a |
| e^x | e^x |
| g(h(x)) | g'(h(x)) · h'(x) (règle de la chaîne) |

La **règle de la chaîne** est au cœur de la *rétropropagation* (backpropagation), l'algorithme qui entraîne les réseaux de neurones.

## Plusieurs variables : le gradient

Quand f dépend de plusieurs paramètres, on calcule une **dérivée partielle** pour chacun, en faisant comme si les autres étaient constants. Le vecteur de toutes les dérivées partielles est le **gradient**. Il pointe dans la direction où f augmente le plus vite.

In [ ]:
import numpy as np

def g(p):
    x, y = p
    return x ** 2 + 3 * y ** 2

def gradient(f, p, h=1e-6):
    grad = np.zeros_like(p)
    for i in range(len(p)):
        e = np.zeros_like(p)
        e[i] = h
        grad[i] = (f(p + e) - f(p - e)) / (2 * h)
    return grad

print(gradient(g, np.array([1.0, 2.0])))   # exact : [2x, 6y] = [2, 12]

> **À retenir** : pour minimiser une fonction, on avance dans la direction **opposée** au gradient. C'est toute l'idée de la descente de gradient, la leçon suivante.

### Exercice 1 · Dérivée numérique

Écris `derivee(f, x, h=1e-5)` (différence centrée), puis calcule `pente`, la dérivée de `f(x) = x**3 - 2*x` en x = 2. La valeur exacte est 3·2² − 2 = 10.

<details><summary>Indice</summary>

(f(x + h) - f(x - h)) / (2 * h)
</details>

In [ ]:
def f(x):
    return x ** 3 - 2 * x

def derivee(f, x, h=1e-5):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(pente - 10) < 1e-4, f"pente vaut {pente}, j'attendais 10."
assert abs(derivee(lambda x: x ** 2, 3) - 6) < 1e-4, "Ta fonction derivee doit marcher pour n'importe quelle fonction : la dérivée de x² en 3 vaut 6."
print("Bravo, tout est correct.")

### Exercice 2 · Gradient exact

Pour `perte(a, b) = (a - 1)**2 + (b + 2)**2`, écris `grad_perte(a, b)` qui renvoie le tuple des deux dérivées partielles **exactes** (pas numériques).

<details><summary>Indice</summary>

La dérivée de (a - 1)² par rapport à a est 2(a - 1) ; b est traité comme une constante.
</details>

In [ ]:
def grad_perte(a, b):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = grad_perte(0, 0)
assert tuple(r) == (-2, 4), f"grad_perte(0, 0) renvoie {r}, j'attendais (-2, 4)."
assert tuple(grad_perte(1, -2)) == (0, 0), "Au minimum (1, -2), le gradient doit être nul."
print("Bravo, tout est correct.")

---
## La descente de gradient

La **descente de gradient** est l'algorithme qui entraîne la quasi-totalité des modèles modernes, de la régression linéaire aux grands modèles de langage. Le principe tient en une ligne : on répète

$$
\theta \leftarrow \theta - \eta \, \nabla L(\theta)
$$

θ représente les paramètres, L la fonction de perte (l'erreur), ∇L son gradient, et η (eta) le **taux d'apprentissage** (*learning rate*) : la taille du pas.

*Animation interactive « gradient » : à voir sur la plateforme web.*

## En une dimension

In [ ]:
def L(x):
    return (x - 3) ** 2

def dL(x):
    return 2 * (x - 3)

x = 10.0
eta = 0.1
for etape in range(30):
    x = x - eta * dL(x)
print(round(x, 4))   # proche de 3, le minimum

## Le taux d'apprentissage

C'est l'hyperparamètre le plus important. Trop petit, l'apprentissage est très lent. Trop grand, on saute par-dessus le minimum et l'erreur explose.

In [ ]:
for eta in [0.01, 0.1, 0.9, 1.1]:
    x = 10.0
    for _ in range(30):
        x = x - eta * dL(x)
    print(f"eta = {eta:<5} → x = {x:.3f}")

## Une régression linéaire, à la main

On cherche la droite ŷ = w·x + b qui colle le mieux aux données. La perte est l'**erreur quadratique moyenne** (MSE) :

$$
L(w, b) = \frac{1}{n} \sum_i (w x_i + b - y_i)^2
\qquad
\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)\, x_i
\qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)
$$

In [ ]:
import numpy as np
import plotly.express as px

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 100)
y = 2.5 * x + 4 + rng.normal(0, 2, 100)     # vraie droite : w = 2.5, b = 4, plus du bruit

w, b, eta = 0.0, 0.0, 0.01
pertes = []
for epoque in range(500):
    y_pred = w * x + b
    erreur = y_pred - y
    pertes.append((erreur ** 2).mean())
    w -= eta * 2 * (erreur * x).mean()
    b -= eta * 2 * erreur.mean()

print(f"w = {w:.2f}, b = {b:.2f}, MSE finale = {pertes[-1]:.2f}")
px.line(y=pertes, title="La perte diminue à chaque époque",
        labels={"y": "MSE", "x": "époque"})

Le modèle retrouve presque la vraie droite (w ≈ 2,5, b ≈ 4) à partir des seules données. scikit-learn et PyTorch font exactement cela, en plus rapide et avec plus de paramètres.

> **À retenir** : une **époque** est un passage complet sur les données. Une courbe de perte qui descend puis se stabilise signifie que l'entraînement a convergé. Une courbe qui remonte signale un taux d'apprentissage trop grand.

### Exercice 1 · Minimiser une parabole

Avec une descente de gradient (eta = 0.1, 100 étapes, départ x = 0), trouve le minimum de `L(x) = (x + 4)**2 + 1`. Stocke le résultat final dans `x_min`. Écris toi-même la dérivée `dL`.

<details><summary>Indice</summary>

La dérivée de (x + 4)² + 1 est 2(x + 4). Répète x -= 0.1 * dL(x).
</details>

In [ ]:
def L(x):
    return (x + 4) ** 2 + 1

def dL(x):
    pass

x = 0.0

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert abs(x_min + 4) < 1e-3, f"x_min vaut {x_min}, le minimum est en -4."
assert dL(-4) == 0, "dL(-4) devrait valoir 0."
print("Bravo, tout est correct.")

### Exercice 2 · Régression sur les manchots

Entraîne par descente de gradient une régression `masse ≈ w × nageoire + b` sur les manchots. Les données sont **standardisées** pour que la descente converge vite (`xs`, `ys`). Utilise eta = 0.1 et 200 époques, et remplis la liste `pertes` avec la MSE à chaque époque. Compare ton `w` à la pente de `np.polyfit`.

<details><summary>Indice</summary>

Reprends la boucle de la leçon en remplaçant x et y par xs et ys, et eta par 0.1.
</details>

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("penguins.csv").dropna()
x = df["flipper_length_mm"].to_numpy()
y = df["body_mass_g"].to_numpy()
xs = (x - x.mean()) / x.std()
ys = (y - y.mean()) / y.std()

w, b = 0.0, 0.0
pertes = []

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
assert len(pertes) == 200, f"pertes contient {len(pertes)} valeurs, j'en attendais 200."
assert pertes[-1] < pertes[0], "La perte doit diminuer."
assert abs(w - np.polyfit(xs, ys, 1)[0]) < 1e-3, f"w vaut {w:.4f}, la pente optimale est {np.polyfit(xs, ys, 1)[0]:.4f}."
print("Bravo, tout est correct.")

---
## Projet 3 · Ta régression linéaire de A à Z

Objectif : écrire une **classe** de régression linéaire entraînée par descente de gradient, avec l'interface `fit` / `predict` de scikit-learn. Tu réutilises les classes (semaine 1), NumPy (semaine 2) et les maths de cette semaine.

Ta classe doit gérer **plusieurs** caractéristiques : X est une matrice (n, k), les poids `coef_` un vecteur de k valeurs, et le biais `intercept_` un nombre. Les gradients en version matricielle :

$$
\hat{y} = X w + b \qquad
\nabla_w L = \frac{2}{n} X^\top (\hat{y} - y) \qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)
$$

**À faire :**

- [ ] Copier ta classe dans un fichier regression_gd.py
- [ ] Tracer la courbe de perte dans un notebook
- [ ] Expliquer dans le README ce que représentent coef_ et intercept_
- [ ] Publier sur GitHub

### Exercice 1 · La classe RegressionGD

Complète `RegressionGD`. `fit(X, y)` initialise `coef_` à des zéros et `intercept_` à 0, puis fait `n_epoques` étapes de descente de gradient en remplissant `self.pertes_` (MSE à chaque époque) ; elle renvoie `self`. `predict(X)` renvoie `X @ coef_ + intercept_`.

<details><summary>Indice</summary>

n, k = X.shape. L'erreur est X @ coef_ + intercept_ - y ; le gradient des poids est 2 / n * X.T @ erreur.
</details>

In [ ]:
import numpy as np

class RegressionGD:
    def __init__(self, eta=0.1, n_epoques=500):
        self.eta = eta
        self.n_epoques = n_epoques

    def fit(self, X, y):
        pass

    def predict(self, X):
        pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
rng = np.random.default_rng(5)
X_t = rng.normal(size=(300, 3))
y_t = X_t @ np.array([1.5, -2.0, 0.7]) + 4 + rng.normal(0, 0.05, 300)
m_t = RegressionGD(eta=0.1, n_epoques=500)
assert m_t.fit(X_t, y_t) is m_t, "fit doit renvoyer self."
assert np.allclose(m_t.coef_, [1.5, -2.0, 0.7], atol=0.05), f"coef_ vaut {m_t.coef_}, j'attendais environ [1.5, -2.0, 0.7]."
assert abs(m_t.intercept_ - 4) < 0.05, f"intercept_ vaut {m_t.intercept_}, j'attendais environ 4."
assert len(m_t.pertes_) == 500 and m_t.pertes_[-1] < m_t.pertes_[0], "pertes_ doit contenir une MSE par époque, en baisse."
assert m_t.predict(X_t[:2]).shape == (2,), "predict doit renvoyer une prédiction par ligne."
print("Bravo, tout est correct.")

### Exercice 2 · Appliquer aux manchots

Utilise ta classe (recopie-la au début) pour prédire `body_mass_g` à partir de `flipper_length_mm`, `bill_length_mm` et `bill_depth_mm`, standardisés. Calcule `rmse`, la racine de l'erreur quadratique moyenne **en grammes** sur toutes les données.

<details><summary>Indice</summary>

rmse = np.sqrt(((predictions - y) ** 2).mean()). Avec des données standardisées, eta = 0.1 et 1 000 époques suffisent.
</details>

In [ ]:
import numpy as np
import pandas as pd
# recopie ta classe RegressionGD ici

df = pd.read_csv("penguins.csv").dropna()
cols = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
X = df[cols].to_numpy()
X = (X - X.mean(axis=0)) / X.std(axis=0)
y = df["body_mass_g"].to_numpy()

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert 380 < rmse < 400, f"rmse vaut {rmse:.1f} g ; avec ces 3 variables on obtient environ 390 g. Ton modèle a-t-il convergé ?"
print("Bravo, tout est correct.")